In [1]:
import pandas as pd

STATISTICAL_SIGNIFICANCE = 0.05
EQUIVALENCE_INTERVAL = 5

data_df = pd.read_excel(r'./CAPRI Data.xlsx')

data_df['Condition_Diff'] = data_df['Condition B'] - data_df['Condition A']

data_df['HMA_WMA'] = data_df['HMA/WMA'].astype('category')

data_df

,Sample,RAP,NMAS,HMA/WMA,Condition A,Condition B,Condition_Diff,HMA_WMA
0,1,0,12.5,HMA,41,72,31,HMA
1,2,0,12.5,HMA,77,79,2,HMA
2,3,0,12.5,HMA,69,67,-2,HMA
3,4,15,12.5,WMA,54,74,20,WMA
4,5,15,12.5,WMA,39,34,-5,WMA
5,6,15,12.5,WMA,45,57,12,WMA
6,7,0,19.0,HMA,75,50,-25,HMA
7,8,0,19.0,HMA,62,80,18,HMA
8,9,0,19.0,HMA,39,30,-9,HMA
9,10,15,19.0,WMA,36,42,6,WMA


In [2]:
import plotly.express as px
import plotly.graph_objects as go

equality_line = px.scatter(data_df, x='Condition A', y='Condition B', 
                 hover_data=['Sample', 'RAP', 'NMAS', 'HMA_WMA'],
                 title='Equality Plot: Condition A vs Condition B')

min_val = min(data_df[['Condition A', 'Condition B']].min())
max_val = max(data_df[['Condition A', 'Condition B']].max())
equality_line.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val],
                         mode='lines', name='1:1 line',
                         line=dict(dash='dash', color='red')))

equality_line.show()

In [3]:
from scipy import stats

t_stat, p_value = stats.ttest_1samp(data_df['Condition_Diff'], 0)
print(f"Paired t-test: t = {t_stat:.3f}, p = {p_value:.4f}")
if p_value < STATISTICAL_SIGNIFICANCE:
    print("Reject H0: The conditions are significantly different.")
else:
    print("Fail to reject H0: No significant difference detected.")

Paired t-test: t = 1.506, p = 0.1458
Fail to reject H0: No significant difference detected.


In [ ]:
from statsmodels.stats.weightstats import ttost_paired

tost_p, _, _ = ttost_paired(data_df['Condition A'], data_df['Condition B'], low=-EQUIVALENCE_INTERVAL, upp=EQUIVALENCE_INTERVAL)

print(f"TOST p-value: {tost_p:.4f}")
if tost_p < STATISTICAL_SIGNIFICANCE:
    print(f"The conditions are equivalent within ±{EQUIVALENCE_INTERVAL}.")
else:
    print("Not enough evidence to claim equivalence between Condition A and Condition B.")

TOST p-value: 0.5877
Not enough evidence to claim equivalence.


In [5]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols('Condition_Diff ~ RAP + NMAS + C(HMA_WMA)', data=data_df).fit()
print(model.summary())
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

                            OLS Regression Results                            
Dep. Variable:         Condition_Diff   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                 -0.086
Method:                 Least Squares   F-statistic:                   0.08872
Date:                Sun, 22 Mar 2026   Prob (F-statistic):              0.915
Time:                        14:43:06   Log-Likelihood:                -104.26
No. Observations:                  24   AIC:                             214.5
Df Residuals:                      21   BIC:                             218.0
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.7404     20.52

,sum_sq,df,F,PR(>F)
C(HMA_WMA),30.375000,1.0,0.076537,0.784751
RAP,30.375000,1.0,0.076537,0.784751
NMAS,40.041667,1.0,0.100894,0.753896
Residual,8334.208333,21.0,NaN,NaN


In [6]:
model_full = ols('Condition_Diff ~ RAP * NMAS * C(HMA_WMA)', data=data_df).fit()
print(model_full.summary())
anova_full = sm.stats.anova_lm(model_full, typ=2)
print("\nANOVA with all interactions:")
anova_full

                            OLS Regression Results                            
Dep. Variable:         Condition_Diff   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                 -0.133
Method:                 Least Squares   F-statistic:                   0.09776
Date:                Sun, 22 Mar 2026   Prob (F-statistic):              0.960
Time:                        14:43:06   Log-Likelihood:                -104.18
No. Observations:                  24   AIC:                             216.4
Df Residuals:                      20   BIC:                             221.1
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

,sum_sq,df,F,PR(>F)
C(HMA_WMA),-1.917089e-08,1.0,-4.628881e-11,1.000000
RAP,-8.676816e-12,1.0,-2.095048e-14,1.000000
RAP:C(HMA_WMA),3.037500e+01,1.0,7.334152e-02,0.789307
NMAS,4.004107e+01,1.0,9.668058e-02,0.759067
NMAS:C(HMA_WMA),3.562080e-07,1.0,8.600770e-10,0.999977
RAP:NMAS,7.738447e-11,1.0,1.868475e-13,1.000000
RAP:NMAS:C(HMA_WMA),5.104167e+01,1.0,1.232419e-01,0.729216
Residual,8.283167e+03,20.0,NaN,NaN


In [15]:
# Plot interaction between variables
from plotly.subplots import make_subplots

interaction_data = data_df.groupby(['RAP', 'HMA_WMA'], observed=False)['Condition_Diff'].mean().reset_index()

fig = make_subplots(rows=2, cols=3, y_title='Condition Difference',
                    subplot_titles=['RAP × HMA/WMA', 'RAP × NMAS', 'NMAS × HMA/WMA', 'HMA/WMA x RAP', 'NMAS x RAP', 'HMA/WMA x NMAS'])


def plot_var(x: str, y: str, col: int, row=1):
    int1 = data_df.groupby([x, y], observed=False)['Condition_Diff'].mean().reset_index()
    for hma_wma in int1[y].unique():
        subset = int1[int1[y] == hma_wma]
        fig.add_trace(go.Scatter(x=subset[x], y=subset['Condition_Diff'], 
                                mode='lines+markers', name=f'{y}={hma_wma}'), row=row, col=col,)
plot_var('RAP', 'HMA_WMA', 1)
plot_var('RAP', 'NMAS', 2)
plot_var('NMAS', 'HMA_WMA', 3)
plot_var('HMA_WMA', 'RAP', 1, 2)
plot_var('NMAS', 'RAP', 2, 2)
plot_var('HMA_WMA', 'NMAS', 3, 2)


fig.show()

In [8]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

"""5.3 Post‑hoc Comparisons (Tukey HSD)

If a factor has more than two levels (here all have exactly two, so Tukey is not needed), you would run Tukey HSD to see which specific levels differ.
For illustration, suppose NMAS had three levels; the code would be:"""

# Only if NMAS had >2 levels
tukey = pairwise_tukeyhsd(endog=data_df['Condition_Diff'], groups=data_df['NMAS'], alpha=0.05)
print(tukey)

 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower    upper  reject
-----------------------------------------------------
  12.5   19.0   2.5833 0.7486 -13.9255 19.0922  False
-----------------------------------------------------


In [9]:
data_df['RAP_continuous'] = data_df['RAP'].astype(int)
model_ancova = ols('Condition_Diff ~ RAP_continuous + C(NMAS) + C(HMA_WMA)', data=data_df).fit()
model_ancova.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:         Condition_Diff   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                 -0.086
Method:                 Least Squares   F-statistic:                   0.08872
Date:                Sun, 22 Mar 2026   Prob (F-statistic):              0.915
Time:                        14:43:07   Log-Likelihood:                -104.26
No. Observations:                  24   AIC:                             214.5
Df Residuals:                      21   BIC:                             218.0
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             5.7083      7.043      0.810      0.427      -8.939      20.356
C(NMAS)[T.19.0]       2.5833      8.133      0.318      0.754     -14.330      19.497
C(HMA_WMA)[T.WMA]    -0.0100      0.036     -0.277      0.785      -0.085       0.065
RAP_continuous       -0.1493      0.540     -0.277      0.785      -1.272       0.973
==============================================================================
Omnibus:                        0.694   Durbin-Watson:                   2.228
Prob(Omnibus):                  0.707   Jarque-Bera (JB):                0.702
Skew:                          -0.347   Prob(JB):                        0.704
Kurtosis:                       2.530   Cond. No.                     1.49e+17
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 1.23e-31. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""